In [ ]:
# Example 1: Define a simple @op and @job, then execute it in-process.
from dagster import op, job


@op
def hello_op():
    return "hello dagster"


@op
def shout_op(s: str) -> str:
    return s.upper() + "!"


@job
def hello_job():
    shout_op(hello_op())


result = hello_job.execute_in_process()
assert result.success
assert result.output_for_node("shout_op") == "HELLO DAGSTER!"
print("job ok:", result.output_for_node("shout_op"))


In [ ]:
# Example 2: Software-defined assets via @asset + materialize.
from dagster import asset, materialize


@asset
def numbers():
    return [1, 2, 3, 4, 5]


@asset
def total(numbers):
    return sum(numbers)


result = materialize([numbers, total])
assert result.success
total_value = result.output_for_node("total")
assert total_value == 15
print("materialized total =", total_value)


In [ ]:
# Example 3: Configurable ops using Config (Pydantic-based).
from dagster import op, job, Config


class GreetConfig(Config):
    name: str = "world"


@op
def greet(config: GreetConfig) -> str:
    return f"hello, {config.name}"


@job
def greet_job():
    greet()


result = greet_job.execute_in_process(run_config={"ops": {"greet": {"config": {"name": "dagster"}}}})
assert result.success
assert result.output_for_node("greet") == "hello, dagster"
print(result.output_for_node("greet"))


In [ ]:
# Example 4: Resources injected into an op.
from dagster import op, job, ConfigurableResource


class GreeterResource(ConfigurableResource):
    prefix: str = "Hi"

    def greet(self, who: str) -> str:
        return f"{self.prefix}, {who}"


@op
def greet_with_resource(greeter: GreeterResource) -> str:
    return greeter.greet("dagster")


@job(resource_defs={"greeter": GreeterResource(prefix="Hello")})
def res_job():
    greet_with_resource()


result = res_job.execute_in_process()
assert result.success
assert result.output_for_node("greet_with_resource") == "Hello, dagster"
print(result.output_for_node("greet_with_resource"))


In [ ]:
# Example 5: Definitions object aggregates assets + jobs (modern entry point).
from dagster import Definitions, asset, op, job


@asset
def my_asset():
    return 42


@op
def my_op():
    return "done"


@job
def my_job():
    my_op()


defs = Definitions(assets=[my_asset], jobs=[my_job])

# dagster 1.13: asset enumeration on Definitions goes through resolve_all_asset_keys().
asset_keys = [k.to_user_string() for k in defs.resolve_all_asset_keys()]
assert "my_asset" in asset_keys

loaded_job = defs.get_job_def("my_job")
result = loaded_job.execute_in_process()
assert result.success
print("asset keys:", asset_keys, "| job ok:", result.success)
